In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from huggingface_hub import login
login("hf_********************")

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install trl
!pip install bitsandbytes

In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer

import torch

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

OUTPUT_DIR = "./outputs/qforge-qwen"

MAX_SEQ_LEN = 4096

# =========================================================
# LOAD DATASET
# =========================================================

dataset = load_dataset(
    "mohd-musheer/qforge-v1",
    split="train"
)

print(dataset)

# =========================================================
# FORMAT DATA
# =========================================================

def format_sample(sample):

    text = f"""<|im_start|>system
You are QForge, an expert software engineering and code reasoning assistant.
<|im_end|>

<|im_start|>user
Instruction:
{sample['instruction']}

Input:
{sample['input']}
<|im_end|>

<|im_start|>assistant
{sample['output']}
<|im_end|>
"""

    return {"text": text}

dataset = dataset.map(format_sample)

# =========================================================
# TOKENIZER
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

# =========================================================
# QUANTIZATION
# =========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# =========================================================
# MODEL
# =========================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

model = prepare_model_for_kbit_training(model)

# =========================================================
# LORA
# =========================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# =========================================================
# TRAINING ARGS
# =========================================================



Dataset({
    features: ['source', 'task_type', 'instruction', 'input', 'output'],
    num_rows: 5264
})


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [2]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=1,

    logging_steps=10,

    save_strategy="epoch",

    optim="paged_adamw_8bit",

    bf16=False,
    fp16=False,

    max_grad_norm=0.3,

    lr_scheduler_type="cosine",

    warmup_steps=100,

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================

training_args.dataset_text_field = "text"
training_args.max_seq_length = MAX_SEQ_LEN

trainer = SFTTrainer(
    model=model,

    train_dataset=dataset,

    processing_class=tokenizer,

    args=training_args
)



In [3]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and br

Step,Training Loss
10,1.592485
20,1.558736
30,1.366901
40,1.236493
50,1.090185
60,1.188126
70,1.068848
80,1.051171
90,1.068200
100,1.074805


TrainOutput(global_step=658, training_loss=1.0902750586303895, metrics={'train_runtime': 14755.3677, 'train_samples_per_second': 0.357, 'train_steps_per_second': 0.045, 'total_flos': 6.882623888257843e+16, 'train_loss': 1.0902750586303895})

In [4]:
trainer.model.save_pretrained(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)

print("TRAINING COMPLETE")

TRAINING COMPLETE


In [5]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id="mohd-musheer/qforge-qwen-adapter",
    repo_type="model"
)

print("ADAPTER UPLOADED")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

ADAPTER UPLOADED


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"

ADAPTER_PATH = OUTPUT_DIR

MERGED_PATH = "./merged-qforge"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

# load adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

# merge
model = model.merge_and_unload()

# save merged
model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print("MERGED MODEL SAVED")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MERGED MODEL SAVED


In [7]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

!cmake -B build
!cmake --build build --config Release -j 4

Cloning into 'llama.cpp'...
remote: Enumerating objects: 95200, done.
remote: Counting objects: 100% (349/349), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 95200 (delta 229), reused 158 (delta 136), pack-reused 94851 (from 5)
Receiving objects: 100% (95200/95200), 393.87 MiB | 36.48 MiB/s, done.
Resolving deltas: 100% (67674/67674), done.
/kaggle/working/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler iden

In [8]:
!python convert_hf_to_gguf.py \
/kaggle/working/merged-qforge \
--outfile /kaggle/working/qforge-f16.gguf \
--outtype f16

INFO:hf-to-gguf:Loading model: merged-qforge
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {2048, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {11008, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float1

In [9]:
!./build/bin/llama-quantize \
/kaggle/working/qforge-f16.gguf \
/kaggle/working/qforge-q4.gguf \
Q4_K_M

llama_print_build_info: build = 9274 (52fb93a2b)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/working/qforge-f16.gguf' to '/kaggle/working/qforge-q4.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 434 tensors from /kaggle/working/qforge-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:                    

In [15]:
!./build/bin/llama-quantize \
/kaggle/working/qforge-f16.gguf \
/kaggle/working/qforge-q5.gguf \
Q5_K_M

llama_print_build_info: build = 9274 (52fb93a2b)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/working/qforge-f16.gguf' to '/kaggle/working/qforge-q5.gguf' as Q5_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 434 tensors from /kaggle/working/qforge-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:                    

In [16]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="/kaggle/working/qforge-q5.gguf",
    path_in_repo="qforge-q5.gguf",
    repo_id="mohd-musheer/qforge-qwen-adapter",
    repo_type="model"
)

print("UPLOAD COMPLETE")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

UPLOAD COMPLETE


In [13]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="/kaggle/working/qforge-q4.gguf",
    path_in_repo="qforge-q4.gguf",
    repo_id="mohd-musheer/qforge-qwen-adapter",
    repo_type="model"
)

print("UPLOAD COMPLETE")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

UPLOAD COMPLETE


In [14]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="/kaggle/working/qforge-f16.gguf",
    path_in_repo="qforge-f16.gguf",
    repo_id="mohd-musheer/qforge-qwen-adapter",
    repo_type="model"
)

print("UPLOAD COMPLETE")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

UPLOAD COMPLETE
